# LiteLLM Agent Tests

Portable agent parity tests for the LiteLLM provider using filesystem storage and an OpenAI-backed model.


In [ ]:
import asyncio
import base64
import json
import os
import shutil
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

from agent_base.core.config import PendingToolRelay
from agent_base.core.messages import Message
from agent_base.core.types import ContentBlockType, ImageContent, TextContent, ToolResultContent
from agent_base.providers.litellm import (
    CompactionConfig,
    ExternalizationConfig,
    LiteLLMAgent,
    LiteLLMConfig,
    LiteLLMMessageFormatter,
)
from agent_base.storage import create_adapters
from agent_base.tools import ToolRegistry, tool

load_dotenv('../../../.env')

MODEL = os.getenv('LITELLM_AGENT_TEST_MODEL_PRIMARY', 'openai/gpt-4o-mini')
REASONING_MODEL = os.getenv('LITELLM_AGENT_TEST_MODEL_REASONING', '')
ENABLE_REASONING = os.getenv('LITELLM_ENABLE_REASONING_TEST') == '1'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY must be set in .env.'

TEST_DATA_ROOT = Path('.context/litellm_agent_test_data')
if TEST_DATA_ROOT.exists():
    shutil.rmtree(TEST_DATA_ROOT)
TEST_DATA_ROOT.mkdir(parents=True, exist_ok=True)

def make_adapters(name: str):
    return create_adapters('filesystem', base_path=str(TEST_DATA_ROOT / name))

def collect_final_text(message: Message) -> str:
    return ' '.join(block.text for block in message.content if hasattr(block, 'text')).strip()

def parse_jsonish(text: str):
    candidate = text.strip()
    if candidate.startswith('```'):
        lines = candidate.splitlines()
        if lines and lines[0].startswith('```'):
            lines = lines[1:]
        if lines and lines[-1].strip() == '```':
            lines = lines[:-1]
        candidate = '\n'.join(lines).strip()
    return json.loads(candidate)

def collect_tool_result_refs(messages):
    refs = []
    for msg in messages:
        for block in msg.content:
            tool_result = getattr(block, 'tool_result', None)
            if not isinstance(tool_result, list):
                continue
            for inner in tool_result:
                text = getattr(inner, 'text', '')
                if text.startswith('Tool result content externalized to .context/'):
                    refs.append(text)
    return refs

async def run_with_stream(agent, prompt, formatter: str = 'json'):
    queue = asyncio.Queue()
    events = []
    async def drain():
        while True:
            try:
                item = await asyncio.wait_for(queue.get(), timeout=0.25)
                events.append(item)
            except asyncio.TimeoutError:
                if task.done():
                    break
        while not queue.empty():
            events.append(queue.get_nowait())
    task = asyncio.create_task(agent.run_stream(prompt, queue, formatter))
    await drain()
    result = await task
    return result, events

class StreamDemo:
    def __init__(self, agent, prompt):
        self.agent = agent
        self.prompt = prompt
        self.queue = asyncio.Queue()
        self.task = asyncio.create_task(agent.run_stream(prompt, self.queue))
        self.events = []

    async def wait_started(self):
        try:
            item = await asyncio.wait_for(self.queue.get(), timeout=10)
            self.events.append(item)
        except asyncio.TimeoutError:
            pass

    async def abort(self):
        await self.agent.abort()
        return await self.task

    async def steer(self, instruction: str):
        result = await self.agent.steer(instruction)
        await self.task
        return result

async def start_demo(agent, prompt):
    demo = StreamDemo(agent, prompt)
    await demo.wait_started()
    return demo

formatter = LiteLLMMessageFormatter()
pprint({'model': MODEL, 'reasoning_model': REASONING_MODEL or '<none>'})


## Tool Schema Tests


In [ ]:
@tool
def add(a: int, b: int) -> str:
    """Add two integers."""
    return str(a + b)

@tool
def multiply(a: int, b: int) -> str:
    """Multiply two integers."""
    return str(a * b)

registry = ToolRegistry()
registry.register_tools([add, multiply])
schemas = registry.get_schemas()
wire_schemas = formatter.format_tool_schemas(schemas)
pprint(wire_schemas)
assert wire_schemas[0]['type'] == 'function'
assert wire_schemas[0]['function']['name'] == 'add'
assert wire_schemas[1]['function']['name'] == 'multiply'


## Basic Agent With Tools


In [ ]:
basic_adapters = make_adapters('basic')
basic_agent = LiteLLMAgent(
    system_prompt='Always use the provided math tools for arithmetic.',
    model=MODEL,
    tools=[add, multiply],
    config_adapter=basic_adapters[0],
    conversation_adapter=basic_adapters[1],
    run_adapter=basic_adapters[2],
)

await basic_agent.initialize()
basic_result = await basic_agent.run('What is 5 + (10 * 2)? Use tools.')
pprint(basic_result.final_message.to_dict())
assert basic_result.stop_reason == 'end_turn'
assert '25' in basic_result.final_answer
assert basic_result.total_steps >= 2

stream_agent = LiteLLMAgent(system_prompt='Reply briefly and exactly.', model=MODEL)
stream_result, stream_events = await run_with_stream(stream_agent, 'Reply with exactly PONG.')
assert stream_result.stop_reason == 'end_turn'
assert 'PONG' in stream_result.final_answer.upper()
assert stream_events


## Agent Resume / Rehydration


In [ ]:
rehydrated_agent = LiteLLMAgent(
    agent_uuid=basic_agent.agent_uuid,
    tools=[add, multiply],
    config_adapter=basic_adapters[0],
    conversation_adapter=basic_adapters[1],
    run_adapter=basic_adapters[2],
)
await rehydrated_agent.initialize()
resume_result = await rehydrated_agent.run('What math result did you produce previously?')
print(resume_result.final_answer)
assert '25' in resume_result.final_answer or '42' in resume_result.final_answer


## Final Answer Check


In [ ]:
def validate_json_answer(answer: str) -> tuple[bool, str]:
    try:
        data = parse_jsonish(answer)
    except json.JSONDecodeError:
        return False, 'Return valid JSON only with keys name and description. Do not use markdown fences.'
    if isinstance(data, dict) and 'name' in data and 'description' in data:
        return True, answer
    return False, 'Return valid JSON only with keys name and description. Do not use markdown fences.'

validate_adapters = make_adapters('validate')
validate_agent = LiteLLMAgent(
    system_prompt='Return JSON only.',
    model=MODEL,
    final_answer_check=validate_json_answer,
    config_adapter=validate_adapters[0],
    conversation_adapter=validate_adapters[1],
    run_adapter=validate_adapters[2],
)
validate_result = await validate_agent.run('Describe a calculator in JSON with fields name and description.')
print(validate_result.final_answer)
parsed = parse_jsonish(validate_result.final_answer)
assert 'name' in parsed and 'description' in parsed


## Multimodal + Client Tool


In [ ]:
@tool
def get_rate(source: str, target: str) -> str:
    """Get a currency exchange rate."""
    rates = {
        ('USD', 'INR'): 83.1,
        ('INR', 'USD'): 0.012,
        ('USD', 'JPY'): 110.0,
    }
    return f'Rate {source}->{target}: {rates[(source, target)]}'

multimodal_adapters = make_adapters('multimodal')
multimodal_agent = LiteLLMAgent(
    system_prompt='Always use get_rate for currency conversion requests.',
    model=MODEL,
    tools=[get_rate],
    config_adapter=multimodal_adapters[0],
    conversation_adapter=multimodal_adapters[1],
    run_adapter=multimodal_adapters[2],
)
img_b64 = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAHggJ/PchI7wAAAABJRU5ErkJggg=='
multimodal_prompt = Message.user([
    ImageContent(source_type='base64', media_type='image/png', data=img_b64),
    TextContent(text='Use get_rate to convert 10 USD to INR. The image is only extra context.'),
])
multimodal_result = await multimodal_agent.run(multimodal_prompt)
print(multimodal_result.final_answer)
assert 'INR' in multimodal_result.final_answer or '831' in multimodal_result.final_answer or '83.1' in multimodal_result.final_answer


## Frontend Tool Relay / Resume


In [ ]:
@tool(executor='frontend')
def user_confirm(message: str) -> str:
    """Ask the user for confirmation before answering."""
    return 'yes'

relay_adapters = make_adapters('relay')
relay_agent = LiteLLMAgent(
    system_prompt='For every request, first call user_confirm with a short approval question. Do not answer until the tool returns.',
    model=MODEL,
    frontend_tools=[user_confirm],
    config_adapter=relay_adapters[0],
    conversation_adapter=relay_adapters[1],
    run_adapter=relay_adapters[2],
)
relay_result = await relay_agent.run('Say hello to me.')
assert relay_result.stop_reason == 'relay'
assert isinstance(relay_agent.agent_config.pending_relay, PendingToolRelay)
pending_call = relay_agent.agent_config.pending_relay.frontend_calls[0]

rehydrated_relay_agent = LiteLLMAgent(
    agent_uuid=relay_agent.agent_uuid,
    model=MODEL,
    frontend_tools=[user_confirm],
    config_adapter=relay_adapters[0],
    conversation_adapter=relay_adapters[1],
    run_adapter=relay_adapters[2],
)
resume_result = await rehydrated_relay_agent.resume_with_relay_results([
    ToolResultContent(tool_name='user_confirm', tool_id=pending_call.tool_id, tool_result='yes')
])
print(resume_result.final_answer)
assert resume_result.stop_reason == 'end_turn'
assert 'Hello' in resume_result.final_answer


## Subagent Orchestration


In [ ]:
@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search flights between two cities."""
    return f'Flight from {origin} to {destination} on {date}: $1200.'

@tool
def search_hotels(city: str, checkin: str, checkout: str) -> str:
    """Search hotels in a city."""
    return f'Hotel in {city} from {checkin} to {checkout}: $200/night.'

subagent_adapters = make_adapters('subagents')
flight_agent = LiteLLMAgent(
    system_prompt='Always use search_flights.',
    description='Finds flight options.',
    model=MODEL,
    tools=[search_flights],
    config_adapter=subagent_adapters[0],
    conversation_adapter=subagent_adapters[1],
    run_adapter=subagent_adapters[2],
)
hotel_agent = LiteLLMAgent(
    system_prompt='Always use search_hotels.',
    description='Finds hotel options.',
    model=MODEL,
    tools=[search_hotels],
    config_adapter=subagent_adapters[0],
    conversation_adapter=subagent_adapters[1],
    run_adapter=subagent_adapters[2],
)
orchestrator = LiteLLMAgent(
    system_prompt='Delegate flights to flight_finder and hotels to hotel_finder, then synthesize a travel plan.',
    model=MODEL,
    subagents={'flight_finder': flight_agent, 'hotel_finder': hotel_agent},
    config_adapter=subagent_adapters[0],
    conversation_adapter=subagent_adapters[1],
    run_adapter=subagent_adapters[2],
)
subagent_result = await orchestrator.run('Plan a Tokyo trip with a New York flight on 2025-03-15 and a hotel from 2025-03-15 to 2025-03-20.')
print(subagent_result.final_answer)
assert subagent_result.stop_reason == 'end_turn'
assert 'Tokyo' in subagent_result.final_answer
assert subagent_result.total_steps >= 1


## Context Externalization


In [ ]:
@tool
def large_result_tool(topic: str) -> str:
    """Return a very large result payload."""
    return f'{topic}: ' + ('detail ' * 4000)

compact_adapters = make_adapters('externalize')
strict_compaction = CompactionConfig(threshold_tokens=2000, preserve_recent_tokens=500)
strict_externalization = ExternalizationConfig(max_prompt_tokens=1200, max_tool_result_tokens=1000, max_combined_tool_result_tokens=1200)

agent_compact = LiteLLMAgent(
    system_prompt='You are a helpful assistant.',
    model=MODEL,
    tools=[large_result_tool],
    compaction_config=strict_compaction,
    externalization_config=strict_externalization,
    config_adapter=compact_adapters[0],
    conversation_adapter=compact_adapters[1],
    run_adapter=compact_adapters[2],
)
await agent_compact.initialize()
big_prompt = 'Please summarize the following text: ' + ('alpha beta gamma ' * 2000)
prompt_result, _ = await run_with_stream(agent_compact, big_prompt)
print(prompt_result.final_answer[:200])
sandbox_dir = Path(agent_compact._sandbox.root)
prompt_refs = [
    block.text
    for msg in agent_compact.agent_config.context_messages
    for block in msg.content
    if getattr(block, 'text', '').startswith('Prompt content externalized to .context/')
]
assert prompt_refs
prompt_path = prompt_refs[0].split(' to ', 1)[1].split('. Read', 1)[0]
assert (sandbox_dir / prompt_path).exists()

tool_agent = LiteLLMAgent(
    system_prompt='Always use large_result_tool when asked about a topic. After the tool returns, summarize briefly.',
    model=MODEL,
    tools=[large_result_tool],
    compaction_config=strict_compaction,
    externalization_config=strict_externalization,
    config_adapter=compact_adapters[0],
    conversation_adapter=compact_adapters[1],
    run_adapter=compact_adapters[2],
)
tool_result = await tool_agent.run('Use large_result_tool for the topic ocean currents and then summarize it.')
print(tool_result.final_answer[:200])
tool_refs = collect_tool_result_refs(tool_agent.agent_config.context_messages)
assert tool_refs
tool_path = tool_refs[0].split(' to ', 1)[1].split('. Read', 1)[0]
assert (Path(tool_agent._sandbox.root) / tool_path).exists()


## Abort / Steer


In [ ]:
abort_agent = LiteLLMAgent(system_prompt='Be very detailed and verbose.', model=MODEL)
abort_demo = await start_demo(abort_agent, 'Explain in great detail how to add 5 and 7.')
abort_result = await abort_demo.abort()
assert abort_result.stop_reason == 'aborted'
assert abort_result.final_message.role.value == 'assistant'

steer_agent = LiteLLMAgent(system_prompt='Be very detailed and verbose.', model=MODEL)
steer_demo = await start_demo(steer_agent, 'Explain in great detail how to add 5 and 7.')
steer_result = await steer_demo.steer('Give only the numeric answer.')
print(steer_result.final_answer)
assert steer_result.stop_reason == 'end_turn'
assert '12' in steer_result.final_answer

@tool
async def slow_calculate(expression: str, delay_seconds: int = 5) -> str:
    """Evaluate a mathematical expression after a delay."""
    await asyncio.sleep(delay_seconds)
    return str(eval(expression, {'__builtins__': {}}, {}))

tool_abort_agent = LiteLLMAgent(
    system_prompt='Always use slow_calculate for math questions before answering.',
    model=MODEL,
    tools=[slow_calculate],
)
tool_demo = await start_demo(tool_abort_agent, 'Use slow_calculate to compute 25 * 4 with delay_seconds=5, then explain the result.')
tool_abort_result = await tool_demo.abort()
assert tool_abort_result.stop_reason == 'aborted'
tool_followup = await tool_abort_agent.run('Now answer briefly: what is 25 * 4? Use slow_calculate again.')
assert tool_followup.stop_reason == 'end_turn'
assert '100' in tool_followup.final_answer

relay_abort_adapters = make_adapters('relay_abort')
relay_abort_agent = LiteLLMAgent(
    system_prompt='For every request, first call user_confirm with a short approval question. Do not answer until the tool returns.',
    model=MODEL,
    frontend_tools=[user_confirm],
    config_adapter=relay_abort_adapters[0],
    conversation_adapter=relay_abort_adapters[1],
    run_adapter=relay_abort_adapters[2],
)
relay_pause = await relay_abort_agent.run('Say hello to me.')
assert relay_pause.stop_reason == 'relay'
relay_abort_result = await relay_abort_agent.abort()
assert relay_abort_result.stop_reason == 'aborted'
relay_abort_agent.reconfigure(system_prompt='Answer directly without using tools.')
relay_followup = await relay_abort_agent.run('Say hello directly.')
assert relay_followup.stop_reason == 'end_turn'
assert 'Hello' in relay_followup.final_answer


## Optional Reasoning


In [ ]:
if ENABLE_REASONING and REASONING_MODEL:
    reasoning_agent = LiteLLMAgent(model=REASONING_MODEL, config=LiteLLMConfig(thinking={'type': 'enabled', 'budget_tokens': 256}))
    reasoning_result = await reasoning_agent.run('What is 25 * 37?')
    print(reasoning_result.final_answer)
    assert reasoning_result.final_answer
else:
    print('Skipping reasoning test. Set LITELLM_ENABLE_REASONING_TEST=1 and LITELLM_AGENT_TEST_MODEL_REASONING to enable it.')
